In [ ]:
import numpy as np
from datasets import load_dataset, concatenate_datasets
import os, csv, gc
import pandas as pd
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    GenerationConfig,
)
import wandb
import sys

# Change working directory to project root if running from src
if os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
# Ensure src is in sys.path so evaluation can be imported
if 'src' not in sys.path:
    sys.path.append('src')

# Import helper functions from evaluation.py
from evaluation import get_tokenizer, prepare_compute_metrics


Handle tokenization and preprocessing for mT0. We need to ensure that the tokenizer is updated with any new tokens (like VA roles) and that the preprocessing function correctly formats inputs and labels for seq2seq training.

In [ ]:
# Custom preprocess for mT0 (T5/mT0 doesn't use src_lang or forced_bos_token_id like mBart)
def make_preprocess_mt0(tokenizer_, max_length=1024):
    def preprocess_(batch):
        model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
        for src, tgt in zip(batch["input"], batch["output"]):
            encoded = tokenizer_(src, truncation=True, padding="max_length", max_length=max_length)
            labels = tokenizer_(text_target=tgt, truncation=True, padding="max_length", max_length=max_length)
            
            pad = tokenizer_.pad_token_id
            labels_ids = [tok if tok != pad else -100 for tok in labels["input_ids"]]
            
            model_inputs["input_ids"].append(encoded["input_ids"])
            model_inputs["attention_mask"].append(encoded["attention_mask"])
            model_inputs["labels"].append(labels_ids)
            
        return model_inputs
    return preprocess_


Training pipeline functions

In [ ]:
model_name = "bigscience/mt0-base"
output_dir = "mT0_model/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs('results', exist_ok=True)

def train_mt0(train_langs, srl_type):
    """
    Train mT0 on a specific set of languages and SRL type.
    """
    # 1. Load tokenizer and get custom vocabulary (VA roles)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer = get_tokenizer(tokenizer)  # From evaluation.py
    
    # 2. Load base model and resize embeddings
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))
    
    print("Tokenizer vocab size:", len(tokenizer))
    print("Model embedding size:", model.get_input_embeddings().weight.size(0))
    
    train_name = "_".join(train_langs)
    run_name = f"{srl_type}_{train_name}_mt0_finetune"
    
    wandb.init(project="mt0-srl-finetuning", name=run_name, reinit=True)
    
    # 3. Load Datasets for the train_langs
    train_datasets = []
    val_datasets = []
    
    for lang in train_langs:
        data_files = {
            "train": f"data/linearizations_{srl_type}_Train_{lang}.tsv",
            "val": f"data/linearizations_{srl_type}_Val_{lang}.tsv"
        }
        raw_datasets = load_dataset("csv", data_files=data_files, delimiter="\t")
        train_datasets.append(raw_datasets["train"])
        val_datasets.append(raw_datasets["val"])
        
    # Concatenate multiple languages if needed
    combined_train = concatenate_datasets(train_datasets)
    combined_val = concatenate_datasets(val_datasets)
    
    # Shuffle train
    combined_train = combined_train.shuffle(seed=42)
    
    preprocess = make_preprocess_mt0(tokenizer)
    train_ds = combined_train.map(preprocess, batched=True)
    val_ds = combined_val.map(preprocess, batched=True)
    
    # 4. Training Arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"eval_results/{run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        predict_with_generate=True,
        generation_max_length=1024,
        logging_dir="logs",
        report_to=["wandb"],
        num_train_epochs=3,
        save_total_limit=1,
        load_best_model_at_end=True,
    )
    
    # Prepare compute metrics on val to see performance during training
    # We pick the combined validation set for metrics
    compute_metrics_val = prepare_compute_metrics(val_ds, srl_type, train_langs)
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model),
        compute_metrics=compute_metrics_val
    )
    
    # 5. Train
    print(f"Starting training for {run_name}...")
    trainer.train()
    
    # Save best model
    best_model_dir = os.path.join(output_dir, f"{train_name}_{srl_type}_mt0_best")
    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    wandb.finish()
    
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_model_dir


In [ ]:
def evaluate_mt0(train_langs, srl_type):
    """
    Evaluate mT0 on all test languages for a given trained model.
    """
    train_name = "_".join(train_langs)
    run_name = f"{srl_type}_{train_name}_mt0_finetune"
    best_model_dir = os.path.join(output_dir, f"{train_name}_{srl_type}_mt0_best")
    
    # Load best model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(best_model_dir)
    
    wandb.init(project="mt0-srl-finetuning", name=f"{run_name}_eval", reinit=True)
    
    preprocess = make_preprocess_mt0(tokenizer)
    
    all_results = []
    
    # 6. Evaluation on ALL test languages
    for test_lang in ['ZH', 'ES', 'EN', 'FR']:
        test_data_files = {
            "test": f"data/linearizations_{srl_type}_Test_{test_lang}.tsv"
        }
        raw_test = load_dataset("csv", data_files=test_data_files, delimiter="\t")
        test_ds = raw_test["test"].map(preprocess, batched=True)
        
        compute_metrics_test = prepare_compute_metrics(test_ds, srl_type, [test_lang])
        
        # Create an evaluator trainer for the test set
        eval_args = Seq2SeqTrainingArguments(
            output_dir="eval_results/temp",
            per_device_eval_batch_size=8,
            predict_with_generate=True,
            generation_max_length=1024,
            report_to=["wandb"]
        )
        
        evaluator = Seq2SeqTrainer(
            model=model,
            args=eval_args,
            eval_dataset=test_ds,
            processing_class=tokenizer,
            data_collator=DataCollatorForSeq2Seq(tokenizer, model),
            compute_metrics=compute_metrics_test
        )
        
        print(f"Evaluating on {test_lang} test set...")
        test_results = evaluator.evaluate()
        
        row = {
            "srl_type": srl_type,
            "train_lang": train_name,
            "test_lang": test_lang,
            **test_results
        }
        all_results.append(row)
        print(f"Test Results ({srl_type}, Train: {train_name}, Test: {test_lang}): {test_results}")
        
        del evaluator
        gc.collect()
        torch.cuda.empty_cache()
        
    wandb.finish()
    
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    # 7. Save final results
    df = pd.DataFrame(all_results)
    results_path = f"results/mt0_FT_results_{run_name}.csv"
    df.to_csv(results_path, index=False)
    print(f"Evaluation completed. Results saved to {results_path}")
    
    return df


Dependency SRL

In [ ]:
train_mt0(['EN'], 'dependency')

In [ ]:
evaluate_mt0(['EN'], 'dependency')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
train_mt0(['ZH'], 'dependency')

In [ ]:
evaluate_mt0(['ZH'], 'dependency')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
train_mt0(['EN', 'ZH'], 'dependency')

In [ ]:
evaluate_mt0(['EN', 'ZH'], 'dependency')
gc.collect()
torch.cuda.empty_cache()

Span SRL

In [ ]:
train_mt0(['EN'], 'span')

In [ ]:
evaluate_mt0(['EN'], 'span')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
train_mt0(['ZH'], 'span')

In [ ]:
evaluate_mt0(['ZH'], 'span')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
train_mt0(['EN', 'ZH'], 'span')

In [ ]:
evaluate_mt0(['EN', 'ZH'], 'span')
gc.collect()
torch.cuda.empty_cache()